<a href="https://colab.research.google.com/github/MarouaDhm/loan_approval_prediction/blob/main/loan_approval_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [12]:
  # Dataset: Loan Prediction training data (Kaggle), file "train_u6lujuX_CVtuZ9i.csv"
df = pd.read_csv("/train_u6lujuX_CVtuZ9i.csv")   # use your train file name
df.columns = df.columns.str.strip()

In [13]:
print(df.shape)
print(df.columns.tolist())
df.info()
print(df.isnull().sum())

(614, 13)
['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_St

In [14]:
print(df["Loan_Status"].value_counts())
print(df["Loan_Status"].value_counts(normalize=True) * 100)
df.head()

Loan_Status
Y    422
N    192
Name: count, dtype: int64
Loan_Status
Y    68.729642
N    31.270358
Name: proportion, dtype: float64


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [15]:
from sklearn.model_selection import train_test_split

# Drop the identifier (no predictive value)
df = df.drop(columns=["Loan_ID"])

# Encode the target: Y -> 1 (approved), N -> 0 (rejected)
df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0})

X = df.drop(columns=["Loan_Status"])
y = df["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True) * 100)
print(y_test.value_counts())

(491, 11) (123, 11)
Loan_Status
1    68.635438
0    31.364562
Name: proportion, dtype: float64
Loan_Status
1    85
0    38
Name: count, dtype: int64


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Column groups
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"]
cat_cols = ["Gender", "Married", "Dependents", "Education",
            "Self_Employed", "Credit_History", "Property_Area"]

# Numeric: fill missing values with the median, then scale
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical: fill missing values with the most frequent value, then one-hot encode
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Apply each pipeline to its own group of columns
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# Baseline model: preprocessing + Logistic Regression in one pipeline
lr = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

# Everything is fitted on the training set only (no leakage)
lr.fit(X_train, y_train)
pred = lr.predict(X_test)

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

[[22 16]
 [ 1 84]]
              precision    recall  f1-score   support

           0       0.96      0.58      0.72        38
           1       0.84      0.99      0.91        85

    accuracy                           0.86       123
   macro avg       0.90      0.78      0.81       123
weighted avg       0.88      0.86      0.85       123



In [17]:
# pip install imbalanced-learn   # run only if the import below fails
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

models = {
    "LogReg (plain)": Pipeline([
        ("prep", preprocess),
        ("model", LogisticRegression(max_iter=1000))]),
    "LogReg (balanced)": Pipeline([
        ("prep", preprocess),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    # SMOTE is applied to the training folds only (imblearn pipeline)
    "LogReg + SMOTE": ImbPipeline([
        ("prep", preprocess),
        ("smote", SMOTE(random_state=42)),
        ("model", LogisticRegression(max_iter=1000))]),
    "DecisionTree (balanced)": Pipeline([
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(max_depth=4, class_weight="balanced",
                                         random_state=42))]),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print("=====", name, "=====")
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))
    # 5-fold CV on the full data for a more reliable comparison
    cv = cross_val_score(m, X, y, cv=skf, scoring="f1_macro")
    print(f"CV macro-F1: {cv.mean():.3f} +/- {cv.std():.3f}\n")

===== LogReg (plain) =====
[[22 16]
 [ 1 84]]
              precision    recall  f1-score   support

           0       0.96      0.58      0.72        38
           1       0.84      0.99      0.91        85

    accuracy                           0.86       123
   macro avg       0.90      0.78      0.81       123
weighted avg       0.88      0.86      0.85       123

CV macro-F1: 0.725 +/- 0.034

===== LogReg (balanced) =====
[[27 11]
 [10 75]]
              precision    recall  f1-score   support

           0       0.73      0.71      0.72        38
           1       0.87      0.88      0.88        85

    accuracy                           0.83       123
   macro avg       0.80      0.80      0.80       123
weighted avg       0.83      0.83      0.83       123

CV macro-F1: 0.690 +/- 0.034

===== LogReg + SMOTE =====
[[26 12]
 [ 8 77]]
              precision    recall  f1-score   support

           0       0.76      0.68      0.72        38
           1       0.87      0.91   